In [1]:
#1. Import Libraries

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
#Load the Dataset

In [4]:
dataset = pd.read_csv('Mall_Customers.csv')

In [5]:
#Display the dataset

In [6]:
dataset

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40
...,...,...,...,...,...
195,196,Female,35,120,79
196,197,Female,45,126,28
197,198,Male,32,126,74
198,199,Male,32,137,18


In [7]:
#Select annual income and spending score

In [8]:
# Select all customers and the two clustering features
X = dataset.iloc[:, [3, 4]].values

print("Shape:", X.shape)
print("First five customers:")
print(X[:5])

Shape: (200, 2)
First five customers:
[[15 39]
 [15 81]
 [16  6]
 [16 77]
 [17 40]]


In [9]:
#Scale the features

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Shape:", X_scaled.shape)
print("First five scaled rows:")
print(X_scaled[:5])

Shape: (200, 2)
First five scaled rows:
[[-1.73899919 -0.43480148]
 [-1.73899919  1.19570407]
 [-1.70082976 -1.71591298]
 [-1.70082976  1.04041783]
 [-1.66266033 -0.39597992]]


In [11]:
#Create and train the OPTICS model

In [12]:
#| `min_samples=10` | Uses 10 points, Number of neighbouring points, including itself, needed for the core-density criterion 
#| `max_eps=np.inf` | Explore all neighbourhood distance scales
#| `cluster_method="xi"` | Extract clusters using changes in reachability
#| `xi=0.05` | Steepness threshold for identifying cluster boundaries
#| `min_cluster_size=0.05` | Minimum cluster size: 5% of 200 customers = 10

In [13]:
from sklearn.cluster import OPTICS

# Starting settings for practice
OP = OPTICS(min_samples=10, 
            max_eps=np.inf, 
            cluster_method="xi", 
            xi=0.05, 
            min_cluster_size=0.05
           )

# Train once and obtain the cluster labels
y_OP = OP.fit_predict(X_scaled)


In [14]:
#Display the labels
#-1 represents unassigned/noise customers. Other numbers identify clusters.

In [15]:
print("Cluster labels:")
print(y_OP)

print("\nUnique labels, including noise:")
print(np.unique(y_OP))

Cluster labels:
[-1  3  0  3 -1  3  0 -1  0  3  0 -1  0  3  0  3 -1  3 -1 -1 -1  3  0  3
  0  3 -1 -1 -1  3  0  3  0  3  0  3  0 -1  0 -1 -1  3 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1 -1
  1  1  1  1 -1  1 -1 -1  1  1 -1  2  1  2 -1 -1 -1  2 -1  2  2  2  2  2
  2  2  2  2  2  2 -1  2  2  2  2  2  2  2  2  2  2  2  2  2  2 -1  2  2
 -1  2 -1 -1 -1  5 -1 -1  4  5  4  5 -1 -1  4 -1  4  5  4  5  4 -1 -1 -1
  4 -1 -1  5 -1 -1  4 -1  4  5  4 -1  4  5  4  5 -1  5  4 -1 -1  5 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1]

Unique labels, including noise:
[-1  0  1  2  3  4  5]


In [16]:
#Count clusters and noise correctly

In [17]:
# Remove -1 before counting clusters
cluster_ids = sorted(set(y_OP) - {-1})

n_clusters = len(cluster_ids)
n_noise = int(np.sum(y_OP == -1))

print("Number of clusters:", n_clusters)
print("Noise customers:", n_noise)
print("Clustered customers:", len(y_OP) - n_noise)
print("Total customers:", len(y_OP))
print("Noise percentage:",round(n_noise / len(y_OP) * 100, 2))

Number of clusters: 6
Noise customers: 94
Clustered customers: 106
Total customers: 200
Noise percentage: 47.0


In [18]:
#Add the results to your dataset

In [19]:
dataset["OPTICS_Cluster"] = y_OP

print("Customers:", len(dataset))
print("Cluster labels:", len(y_OP))

display(dataset.head(10))

Customers: 200
Cluster labels: 200


,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100),OPTICS_Cluster
0,1,Male,19,15,39,-1
1,2,Male,21,15,81,3
2,3,Female,20,16,6,0
3,4,Female,23,16,77,3
4,5,Female,31,17,40,-1
5,6,Female,22,17,76,3
6,7,Female,35,18,6,0
7,8,Female,23,18,94,-1
8,9,Male,64,19,3,0
9,10,Female,30,19,72,3


In [20]:
#Summarize each group
#row -1 appears, it summarizes the unassigned customers collectively. It is not an additional cluster.

In [21]:
summary = dataset.groupby("OPTICS_Cluster").agg(
    Customers=("CustomerID", "count"),
    Average_Income=("Annual Income (k$)", "mean"),
    Average_Spending=("Spending Score (1-100)", "mean")
).round(2)

display(summary)

,Customers,Average_Income,Average_Spending
OPTICS_Cluster,,,
-1,94,69.65,54.41
0,13,25.54,10.85
1,23,49.00,52.22
2,31,61.90,46.84
3,15,24.53,78.73
4,13,75.77,9.15
5,11,75.82,75.55


In [22]:
#Inspect the noise customers
#Keep these customers for separate review.

In [23]:
noise_customers = dataset[
    dataset["OPTICS_Cluster"] == -1
].copy()

display(noise_customers)

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100),OPTICS_Cluster
0,1,Male,19,15,39,-1
4,5,Female,31,17,40,-1
7,8,Female,23,18,94,-1
11,12,Female,35,19,99,-1
16,17,Female,35,21,35,-1
...,...,...,...,...,...,...
195,196,Female,35,120,79,-1
196,197,Female,45,126,28,-1
197,198,Male,32,126,74,-1
198,199,Male,32,137,18,-1


In [53]:
#Compare OPTICS parameter combinations
#This produces 27 experiments. NaN means a silhouette score could not be calculated for that result.
#The comparison does not change your original OP model or y_OP labels. 
#To adopt a different combination, change Step 6 and rerun Steps 6–13 before exporting.

In [54]:
#Export your results

In [25]:
# Customer records with their cluster labels
dataset.to_csv(
    "OPTICS_Customer_Results.csv",
    index=False
)

# Summary of the selected model
summary.to_csv(
    "OPTICS_Cluster_Summary.csv"
)

# Customers requiring separate review
noise_customers.to_csv(
    "OPTICS_Noise_Customers.csv",
    index=False
)


print("Results saved in the notebook's working folder.")

Results saved in the notebook's working folder.
